# Product Catalog Similarity Recommender

**Objective:** Recommend similar catalog items from product text and metadata without user labels or purchase outcomes.

The deterministic catalog keeps the full recommendation workflow executable without external data or user tracking.


## 1. Setup and reproducibility


In [1]:
import platform
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import sklearn

warnings.filterwarnings("ignore")
RANDOM_STATE = 42
rng = np.random.default_rng(RANDOM_STATE)
sns.set_theme(style="whitegrid", palette="deep")

print(f"Python: {platform.python_version()}")
print(f"pandas: {pd.__version__} | NumPy: {np.__version__} | scikit-learn: {sklearn.__version__}")
print(f"Random seed: {RANDOM_STATE}")


Python: 3.12.13
pandas: 2.2.3 | NumPy: 2.3.5 | scikit-learn: 1.8.0
Random seed: 42


## 2. Build a product catalog with text and metadata


In [2]:
CATEGORIES = {
    "laptop": ["processor", "ram", "ssd", "display", "keyboard", "portable", "battery", "developer"],
    "headphones": ["audio", "wireless", "noise", "bass", "microphone", "comfort", "battery", "music"],
    "camera": ["sensor", "lens", "video", "stabilization", "autofocus", "photo", "creator", "resolution"],
    "fitness": ["workout", "tracking", "heart", "steps", "sleep", "waterproof", "health", "training"],
    "home_office": ["desk", "chair", "ergonomic", "monitor", "lighting", "workspace", "adjustable", "support"],
}
rows = []
for category, words in CATEGORIES.items():
    for item_id in range(55):
        selected = rng.choice(words, size=6, replace=True).tolist()
        price_band = rng.choice(["budget", "midrange", "premium"], p=[0.3, 0.45, 0.25])
        selected += [price_band, rng.choice(["compact", "standard", "professional"])]
        rows.append({"product_id": f"{category[:3]}-{item_id:03d}", "category": category, "price_band": price_band, "description": " ".join(selected)})
catalog = pd.DataFrame(rows).sample(frac=1, random_state=RANDOM_STATE).reset_index(drop=True)
print(f"Catalog size: {len(catalog):,} products")
display(catalog.head())


Catalog size: 275 products


,product_id,category,price_band,description
0,lap-030,laptop,premium,ssd processor ram developer display display pr...
1,cam-029,camera,premium,stabilization sensor creator photo sensor lens...
2,fit-030,fitness,midrange,heart heart sleep sleep sleep steps midrange s...
3,cam-016,camera,midrange,autofocus sensor photo video photo autofocus m...
4,fit-050,fitness,budget,tracking heart sleep training tracking trackin...


## 3. Vectorize and retrieve similar products


In [3]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.neighbors import NearestNeighbors

vectorizer = TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True)
vectors = vectorizer.fit_transform(catalog["description"])
model = NearestNeighbors(n_neighbors=7, metric="cosine").fit(vectors)
query_index = int(catalog.index[catalog["category"] == "camera"][0])
distances, indices = model.kneighbors(vectors[query_index])
recommendations = catalog.iloc[indices[0]].copy()
recommendations["cosine_similarity"] = 1 - distances[0]
print(f"Query: {catalog.loc[query_index, 'product_id']} | {catalog.loc[query_index, 'description']}")
display(recommendations.round(4))


Query: cam-029 | stabilization sensor creator photo sensor lens premium standard


,product_id,category,price_band,description,cosine_similarity
1,cam-029,camera,premium,stabilization sensor creator photo sensor lens...,1.0000
9,cam-032,camera,budget,sensor creator photo photo stabilization creat...,0.4498
44,cam-003,camera,budget,stabilization sensor sensor sensor sensor crea...,0.4495
106,cam-034,camera,premium,video photo autofocus sensor creator stabiliza...,0.4093
54,cam-047,camera,premium,creator autofocus creator photo stabilization ...,0.3682
60,cam-037,camera,midrange,resolution creator lens photo sensor photo mid...,0.3663
122,cam-006,camera,midrange,photo lens lens autofocus photo sensor midrang...,0.3503


## 4. Discover catalog groups without category labels


In [4]:
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score, silhouette_score

selection = []
for k in range(2, 8):
    labels = KMeans(n_clusters=k, n_init=30, random_state=RANDOM_STATE).fit_predict(vectors)
    selection.append({"k": k, "cosine_silhouette": silhouette_score(vectors, labels, metric="cosine")})
selection = pd.DataFrame(selection)
display(selection.round(4))
best_k = int(selection.loc[selection["cosine_silhouette"].idxmax(), "k"])
cluster_model = KMeans(n_clusters=best_k, n_init=40, random_state=RANDOM_STATE).fit(vectors)
catalog["cluster"] = cluster_model.labels_
display(pd.crosstab(catalog["category"], catalog["cluster"], margins=True))
print(f"Retrospective ARI against hidden catalog categories: {adjusted_rand_score(catalog['category'], catalog['cluster']):.4f}")


Retrospective ARI against hidden catalog categories: 1.0000


,k,cosine_silhouette
0,2,0.0823
1,3,0.1220
2,4,0.1614
3,5,0.1926
4,6,0.1716
5,7,0.1509


cluster,0,1,2,3,4,All
category,,,,,,
camera,0,55,0,0,0,55
fitness,0,0,0,0,55,55
headphones,0,0,55,0,0,55
home_office,0,0,0,55,0,55
laptop,55,0,0,0,0,55
All,55,55,55,55,55,275


## 5. Coverage and diversity checks


In [5]:
sample_indices = catalog.groupby("category", sort=True).head(1).index
rows = []
for idx in sample_indices:
    distances, indices = model.kneighbors(vectors[idx])
    returned = catalog.iloc[indices[0][1:]]
    rows.append({"query_category": catalog.loc[idx, "category"], "mean_similarity": float((1 - distances[0][1:]).mean()), "unique_categories": int(returned["category"].nunique()), "unique_price_bands": int(returned["price_band"].nunique())})
display(pd.DataFrame(rows).round(4))


,query_category,mean_similarity,unique_categories,unique_price_bands
0,laptop,0.4321,1,2
1,camera,0.3989,1,3
2,fitness,0.4489,1,3
3,home_office,0.4772,1,3
4,headphones,0.3987,1,3


## 6. Findings and limitations

- Content similarity supports cold-start items but cannot learn personal taste without interactions.
- Offline similarity is not evidence of commercial impact.
- Production evaluation should include relevance judgments, diversity, availability, fairness, and online experiments.
